In [ ]:
# !pip install pip nbconvert jupyter pandas tabulate

  Using cached tabulate-0.9.0-py3-none-any.whl.metadata (34 kB)


In [3]:
from logging import Filterer
from pathlib import Path
import pandas as pd
import os

path = Path("..") / "logs" / "test_results.csv"

# read the csv file
df = pd.read_csv(path)

Filtered = df[df['result'] == 'FAIL']

ModuleNotFoundError: No module named 'pandas'

In [ ]:
def get_stdio_and_err(name, evidence_file, failure_stage):
    "expects name to be a string in format: functional/fcntl"
    evidence_path = Path("..") / evidence_file

    std_out = ""
    std_err = ""
    host_out = ""
    host_err = ""
    error_file = ""
    meta_data = ""

    def read_file_if_exists(path):
        if path.exists():
            return path.read_text(encoding="utf-8", errors="replace")
        return ""

    error_file = read_file_if_exists(evidence_path)

    if failure_stage == "runtime":
        artifact_dir = Path("..") / "logs" / "test_stdio" / Path(evidence_file).with_suffix(".exe")
        std_out = read_file_if_exists(artifact_dir / "dandelion.stdout")
        std_err = read_file_if_exists(artifact_dir / "dandelion.stderr")
        host_out = read_file_if_exists(artifact_dir / "host.stdout")
        host_err = read_file_if_exists(artifact_dir / "host.stderr")
        meta_data = read_file_if_exists(artifact_dir / "meta.txt")

    return {"stdout": std_out, "stderr": std_err, ".err": error_file, "host_out": host_out, "host_err": host_err, "meta_data": meta_data}


In [ ]:
rows = []
for index, row in Filtered.iterrows():
    data = get_stdio_and_err(
        str(row["test"]),
        str(row["evidence_file"]),
        str(row["failure_stage"]),
    )
    rows.append({
        "test": str(row["test"]),
        "result": str(row["result"]),
        "failure_stage": str(row["failure_stage"]),
        "evidence_file": str(row["evidence_file"]),
        "stdout": data["stdout"],
        "stderr": data["stderr"],
        ".err": data[".err"],
        "host_out": data["host_out"],
        "host_err": data["host_err"],
        "meta_data": data["meta_data"]
    })

df_new = pd.DataFrame(rows)

# filter into categories
df_segfault = df_new[df_new[".err"].str.contains("Segmentation fault", na=False)]
df_new = df_new[~df_new[".err"].str.contains("Segmentation fault", na=False)]

timed_out = df_new[df_new[".err"].str.contains("timed out", na=False)]
df_new = df_new[~df_new[".err"].str.contains("timed out", na=False)]

empty_stdout = df_new[df_new["stdout"] == ""]
std_out_rows = df_new[df_new["stdout"] != ""]

#TODO analyze stdout

#1. Function not implemented


In [ ]:
from pathlib import Path

stout_appended = ""
report_dir = Path("..") / "report"
report_dir.mkdir(parents=True, exist_ok=True)

# write stout_appended.txt
for _, row in std_out_rows.iterrows():
    for line in row["stdout"].splitlines():
        if line.strip():
            stout_appended += line + "\n"

(report_dir / "stout_appended.txt").write_text(stout_appended)


In [ ]:
import re
from pathlib import Path
from typing import Dict, List, Optional, Tuple

scripts_dir = Path("..") / "scripts"
report_dir = Path("..") / "report"
report_dir.mkdir(parents=True, exist_ok=True)

SOURCE_PREFIX_RE = re.compile(r'^(?:X\s+)?(?P<source>[^:\n]+):(?P<source_line>\d+):\s+')
PREFIX = r'^(?:X\s+)?[^:\n]+:\d+:\s+'
FUNCTION_PATTERNS = [
    # Matches assignment chains like "x = y = fopen(...)".
    re.compile(PREFIX + r'\(?\s*(?:[A-Za-z_][A-Za-z0-9_]*\s*=\s*)+([A-Za-z_][A-Za-z0-9_]*)\s*(?=\()'),
    # Matches FP-mode-prefixed calls like "RN nearbyint(...)".
    re.compile(PREFIX + r'(?:bad fp exception:\s+)?(?:RN|RZ|RU|RD)\s+([A-Za-z_][A-Za-z0-9_]*)\s*(?=\()'),
    # Matches ordinal call descriptions like "1st foo(...)".
    re.compile(PREFIX + r'\d+(?:st|nd|rd|th)\s+([A-Za-z_][A-Za-z0-9_]*)\s*(?=\()'),
    # Matches sizeof-based call descriptions like "sizeof pow(...) want 4 got 8".
    re.compile(PREFIX + r'sizeof\s+([A-Za-z_][A-Za-z0-9_]*)\s*(?=\()'),
    # Matches bracketed prefixes before failed calls like "[thread 1]: pthread_create(...) == 0 failed".
    re.compile(PREFIX + r'(?:\[[^\]]+\]:\s+)+([A-Za-z_][A-Za-z0-9_]*)\s*(?=\().*\bfailed\b'),
    # Matches negated failed calls like "!!feof(...) failed" or "!pthread_join(...) failed".
    re.compile(PREFIX + r'!+\s*([A-Za-z_][A-Za-z0-9_]*)\s*(?=\().*\bfailed\b'),
    # Matches parenthesized function contexts like "failed (blocking sem_wait, ...)".
    re.compile(PREFIX + r'.*\bfailed\b\s*\((?:blocking|non-blocking)\s+([A-Za-z_][A-Za-z0-9_]*)\b'),
    # Matches parenthesized function contexts like "failed (shm_open, seqno)".
    re.compile(PREFIX + r'.*\bfailed\b\s*\(([A-Za-z_][A-Za-z0-9_]*)\s*,'),
    # Matches status lines like "regcomp returned 2".
    re.compile(PREFIX + r'([A-Za-z_][A-Za-z0-9_]*)\s+returned\s+[-+]?\d+\b'),
    # Matches expectation lines like "pthread_create should fail with EAGAIN but failed with 1".
    re.compile(PREFIX + r'([A-Za-z_][A-Za-z0-9_]*)\s+should\s+fail\b'),
    # Matches descriptive result lines like "fgets read back: ...".
    re.compile(PREFIX + r'([A-Za-z_][A-Za-z0-9_]*)\s+read back\s*:'),
    # Matches narrative lines like "pthread_once ran init 0 times".
    re.compile(PREFIX + r'([A-Za-z_][A-Za-z0-9_]*)\s+ran\b'),
    # Matches narrative failures like "fesetenv did not restore upward rounding".
    re.compile(PREFIX + r'([a-z_][A-Za-z0-9_]*)\s+did\s+not\b'),
    # Matches regression text like "ftello is broken before flush".
    re.compile(PREFIX + r'([a-z_][A-Za-z0-9_]*)\s+is\s+broken\b'),
    # Matches OOM-style lines like "malloc was successful".
    re.compile(PREFIX + r'([a-z_][A-Za-z0-9_]*)\s+was\s+successful\b'),
    # Matches mismatch lines like "read 1 instead of 0".
    re.compile(PREFIX + r'([a-z_][A-Za-z0-9_]*)\s+[-+]?\d+\s+instead\s+of\s+[-+]?\d+\b'),
    # Matches status lines like "getrlimit 2: Success".
    re.compile(PREFIX + r'([A-Za-z_][A-Za-z0-9_]*)\s+\d+\s*:'),
    # Matches plain call forms like "fopen(...)".
    re.compile(PREFIX + r'([A-Za-z_][A-Za-z0-9_]*)\s*(?=\()'),
    # Matches plain labels like "malloc:".
    re.compile(PREFIX + r'([A-Za-z_][A-Za-z0-9_]*)\s*:'),
    # Matches lowercase trailing failure lines like "pthread_kill failed".
    re.compile(PREFIX + r'([a-z_][A-Za-z0-9_]{1,})\s+failed\b'),
]


def extract_source_info(line: str) -> Tuple[Optional[str], Optional[int]]:
    match = SOURCE_PREFIX_RE.match(line)
    if not match:
        return None, None
    return match.group("source"), int(match.group("source_line"))



def normalize_function_name(name: str) -> Optional[str]:
    # Drop obvious non-function captures like one-letter locals and macro names.
    if len(name) == 1 and name.islower():
        return None
    if re.fullmatch(r'[A-Z0-9_]+', name):
        return None
    return name



def extract_func_name(line: str) -> Optional[str]:
    for pattern in FUNCTION_PATTERNS:
        match = pattern.search(line)
        if match:
            function_name = normalize_function_name(match.group(1))
            if function_name:
                return function_name
    return None



def last_nonempty_line(text: str) -> str:
    for line in reversed(text.splitlines()):
        if line.strip():
            return line
    return ""



def add_test_level_rows(frame: pd.DataFrame, failure_category: str, rows: List[Dict[str, object]]) -> None:
    for _, row in frame.iterrows():
        raw_line = last_nonempty_line(row[".err"])
        rows.append(
            {
                "test": row["test"],
                "failure_category": failure_category,
                "function_name": None,
                "source": None,
                "source_line": None,
                "raw_line": raw_line,
            }
        )


SUMMARY_LABELS = {
    "function_not_implemented": "Not implemented",
    "stdout_failure": "Implemented - FAILS TESTS",
}


def build_function_summary(
    frame: pd.DataFrame,
    known_stub_functions: Optional[set[str]] = None,
) -> pd.DataFrame:
    summary_rows = frame.loc[
        frame["function_name"].notna() & frame["failure_category"].isin(SUMMARY_LABELS),
        ["function_name", "failure_category"],
    ].copy()
    summary_rows["status"] = summary_rows["failure_category"].map(SUMMARY_LABELS)

    if known_stub_functions:
        stub_rows = pd.DataFrame(
            {
                "function_name": sorted(known_stub_functions),
                "failure_category": "function_not_implemented",
                "status": SUMMARY_LABELS["function_not_implemented"],
            }
        )
        summary_rows = pd.concat([summary_rows, stub_rows], ignore_index=True)

    function_summary_df = (
        summary_rows.groupby("function_name")["status"]
        .agg(
            lambda statuses: "Not implemented"
            if "Not implemented" in set(statuses)
            else "Implemented - FAILS TESTS"
        )
        .reset_index()
        .sort_values(["status", "function_name"])
    )

    return function_summary_df


analysis_rows = []

# Add segfault, timed out and empty stdout rows
add_test_level_rows(df_segfault, "segmentation_fault", analysis_rows)
add_test_level_rows(timed_out, "timed_out", analysis_rows)
add_test_level_rows(empty_stdout, "empty_stdout", analysis_rows)

for _, row in std_out_rows.iterrows():
    for raw_line in row["stdout"].splitlines():
        #skip empty lines
        if not raw_line.strip():
            continue

        source, source_line = extract_source_info(raw_line)
        function_name = extract_func_name(raw_line)
        failure_category = (
            "function_not_implemented"
            if "Function not implemented" in raw_line
            else "stdout_failure"
        )

        analysis_rows.append(
            {
                "test": row["test"],
                "failure_category": failure_category,
                "function_name": function_name,
                "source": source,
                "source_line": source_line,
                "raw_line": raw_line,
            }
        )

analysis_df = pd.DataFrame(
    analysis_rows,
    columns=["test", "failure_category", "function_name", "source", "source_line", "raw_line"],
)
analysis_df["source_line"] = analysis_df["source_line"].astype("Int64")

function_summary_df = build_function_summary(analysis_df)

stout_lines = analysis_df.loc[analysis_df["failure_category"] == "stdout_failure", "raw_line"]

analysis_df.to_csv(report_dir / "failure_line_details.csv", index=False)
function_summary_df.to_csv(report_dir / "function_failure_summary.csv", index=False)


funcs = sorted(function_summary_df["function_name"].unique().tolist())
function_summary_df.head()


uncategorized_lines = analysis_df.loc[
    (analysis_df["failure_category"] == "stdout_failure") & (analysis_df["function_name"].isna()),
    "raw_line"
]

print(len(uncategorized_lines))

uncategorized_lines_string = ""
for line in uncategorized_lines:
    uncategorized_lines_string += line + "\n"

(report_dir / "uncategorized_lines.txt").write_text(uncategorized_lines_string)


In [ ]:
# Outputs written under ../report by the extraction cells above:
# - failure_line_details.csv: one row per observed failure line
# - function_failure_summary.csv: one row per function with a summary status
# - stout_appended.txt: stdout failures excluding "Function not implemented"
# - uncategorized_lines.txt: stdout failures where no function name was extracted


In [ ]:
function_summary_df[function_summary_df["status"] == "Implemented - FAILS TESTS"]


In [ ]:
analysis_df["failure_category"].value_counts()


In [ ]:
stdout_line_count = sum(
    1
    for _, row in std_out_rows.iterrows()
    for line in row["stdout"].splitlines()
    if line.strip()
)

analysis_stdout_count = (
    analysis_df["failure_category"].isin(["stdout_failure", "function_not_implemented"])
).sum()

assert stdout_line_count == analysis_stdout_count

In [ ]:
# save function_summary_df as markdown
function_summary_df.to_markdown(report_dir / "function_failure_summary.md", index=False)

In [ ]:
stub_inventory_path = scripts_dir / "stub_inventory" / "stubbed_functions.csv"
stub_inventory_df = pd.read_csv(stub_inventory_path)
known_stub_functions = set(stub_inventory_df["function"].dropna())

analysis_df["function_name"] = analysis_df["function_name"].replace({
    "memfill": "mmap",
    "vmfill": "mmap",
})

known_stub_mask = (
    analysis_df["failure_category"].eq("stdout_failure")
    & analysis_df["function_name"].isin(known_stub_functions)
)
analysis_df.loc[known_stub_mask, "failure_category"] = "function_not_implemented"

function_summary_df = build_function_summary(analysis_df, known_stub_functions)

analysis_df.to_csv(report_dir / "failure_line_details.csv", index=False)
function_summary_df.to_csv(report_dir / "function_failure_summary.csv", index=False)
function_summary_df.to_markdown(report_dir / "function_failure_summary.md", index=False)

analysis_df["failure_category"].value_counts()


In [ ]:
unique_function_counts = (
    analysis_df.dropna(subset=["function_name"])
    .groupby("failure_category")["function_name"]
    .nunique()
    .reindex(analysis_df["failure_category"].value_counts().index, fill_value=0)
)

unique_function_counts